# CFD Geometry — quick start

Try **CFDGeometry** in the browser (Google Colab) or locally in **VS Code / Cursor**.

| Launch | |
|--------|---|
| Colab | [Open in Colab](https://colab.research.google.com/github/Omokayode/CFDGeometry/blob/main/notebooks/cfd_geometry_quickstart.ipynb) |
| VS Code | Clone [Omokayode/CFDGeometry](https://github.com/Omokayode/CFDGeometry), open this file, use the project `.venv` kernel |

**What you will do:** draw a small study rectangle on a map → download OSM buildings & trees → write aligned `buildings.stl` and `trees.stl`.

Keep the drawn box **small** (roughly city block scale) so download and extrusion finish in a few minutes.


In [ ]:
# Install — run this cell first (no prior cfd_geometry import needed)
import subprocess
import sys
from pathlib import Path

_GIT = "git+https://github.com/Omokayode/CFDGeometry.git@main"
_WIDGETS = (
    "ipywidgets>=7.6,<9",
    "ipyleaflet>=0.17",
    "jupyterlab_widgets>=1.0.5,<4",
    "plotly>=5.18",
)


def _in_colab() -> bool:
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False


def _pip(*args: str) -> None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])


repo = Path.cwd()
if _in_colab():
    # Keep Colab's ipython==7.34; do not install the [notebook] extra here.
    _pip(*_WIDGETS, "--upgrade-strategy", "only-if-needed")
    _pip(f"{_GIT}#egg=cfd-geometry[download]")
elif (repo / "src" / "cfd_geometry").exists():
    _pip("-e", f"{repo}[notebook,download]")
else:
    _pip(f"{_GIT}#egg=cfd-geometry[notebook,download]")

from cfd_geometry.notebook import setup_colab_widgets

if setup_colab_widgets():
    print("Colab widget manager enabled (required for the map).")

import cfd_geometry

print("cfd_geometry", cfd_geometry.__version__)
print("Colab warnings about moviepy/decorator vs ipython are usually harmless.")


In [ ]:
# Optional: center the map on a place name (needs osmnx)
PLACE = "Milwaukee, Wisconsin, USA"  # or set CENTER = (lat, lon) below
CENTER = None  # e.g. (43.0389, -87.9065)

from cfd_geometry.notebook import select_extent

selector = select_extent(place=PLACE if CENTER is None else None, center=CENTER)
selector


In [ ]:
if selector.bbox is None:
    raise RuntimeError(
        "Draw a rectangle on the map, then click 'Use this extent' before running this cell."
    )

bbox = selector.bbox
print(
    f"west={bbox.west:.6f} south={bbox.south:.6f} "
    f"east={bbox.east:.6f} north={bbox.north:.6f}"
)
print(
    f"CLI: cfd-geometry domain -o data --bbox {bbox.west:.6f} {bbox.south:.6f} "
    f"{bbox.east:.6f} {bbox.north:.6f}"
)


In [ ]:
from pathlib import Path

from cfd_geometry.domain import DomainConfig, build_domain

OUTPUT = Path("data")

config = DomainConfig(
    output_dir=OUTPUT,
    bbox=selector.bbox,
    run_download=True,
    download_layers=("buildings", "trees"),
    download_dem=False,
    build_buildings=True,
    build_trees=True,
    build_highways=False,
    build_terrain=False,
    height_source="composite",
)

result = build_domain(config)
result.stl_files


In [ ]:
# Inspect outputs
from pathlib import Path

out = Path("data/output")
if out.exists():
    for p in sorted(out.glob("*.stl")):
        print(p.name, f"{p.stat().st_size / 1024:.1f} KiB")
summary = Path("data/output/domain_summary.json")
if summary.exists():
    print("Summary:", summary)


In [ ]:
# Interactive 3D preview (Plotly) — rotate/zoom in the output
from cfd_geometry.notebook import plot_domain_stls

# Lower max_triangles if the notebook feels slow on large areas
plot_domain_stls(result, layers=("buildings", "trees"), max_triangles=8000)
